
# Phase 11B — Separate Learning of Drift and Diffusion

## Goal

Phase 11A showed that a constrained structured field can fit the mixed dissipative/rotational dynamics and recover the *directions* of the two components, while satisfying the structural constraints exactly.

Phase 11B adds stochastic dynamics:

\[
\boxed{
dz_t =
\left[
-M_\theta\nabla U_\theta(z_t)
+
A_\theta\nabla H_\theta(z_t)
\right]dt
+
G_\theta(z_t)\,dW_t
}
\]

The learning problem is now explicitly split into:

\[
\text{drift } b_\theta(z)
\]

and

\[
\text{diffusion covariance }
D_\theta(z)=G_\theta(z)G_\theta(z)^\top.
\]

We compare:

1. **Plain SDE** — unconstrained MLP drift + learned diffusion;
2. **Structured SDE** — dissipative + rotational drift + the same learned diffusion family.

Both models are trained only from noisy one-step transitions.  
The true drift components and true diffusion are used **only for evaluation**.

### Important identifiability point

The noise factor \(G\) is generally not unique because

\[
GQ(GQ)^\top=GG^\top
\]

for orthogonal \(Q\).

Therefore the scientifically relevant diffusion target is

\[
\boxed{D=GG^\top}
\]

rather than the raw factor \(G\).


In [1]:

# Optional:
# !pip install -U torch numpy pandas matplotlib

import math
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 20260928
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

print("device:", DEVICE)


device: cpu



## 1. Ground-truth drift

We keep the same deterministic system used in Phase 11A:

\[
U^\star(x,y)
=
\frac12x^2+y^2+0.05x^4,
\]

\[
H^\star(x,y)
=
\frac12(x^2+y^2),
\]

\[
M^\star=
\begin{pmatrix}
1 & 0.2\\
0.2 & 0.7
\end{pmatrix},
\qquad
A^\star=
\begin{pmatrix}
0 & -1.25\\
1.25 & 0
\end{pmatrix}.
\]

Thus

\[
b^\star(z)
=
-M^\star\nabla U^\star(z)
+
A^\star\nabla H^\star(z).
\]


In [2]:

M_TRUE = torch.tensor(
    [[1.0, 0.20],
     [0.20, 0.70]],
    dtype=DTYPE,
    device=DEVICE,
)

A_TRUE = torch.tensor(
    [[0.0, -1.25],
     [1.25, 0.0]],
    dtype=DTYPE,
    device=DEVICE,
)

def true_grad_U(z):
    x = z[..., 0]
    y = z[..., 1]
    return torch.stack(
        [x + 0.20*x**3, 2.0*y],
        dim=-1,
    )

def true_grad_H(z):
    return z

def true_drift_components(z):
    gu = true_grad_U(z)
    gh = true_grad_H(z)
    f_diss = -(gu @ M_TRUE.T)
    f_rot = gh @ A_TRUE.T
    return f_diss, f_rot

def true_drift(z):
    fd, fr = true_drift_components(z)
    return fd + fr

print("eig(M_true):", torch.linalg.eigvalsh(M_TRUE).detach().cpu().numpy())
print("skew error A_true:", torch.norm(A_TRUE + A_TRUE.T).item())


eig(M_true): [0.59999996 1.1       ]
skew error A_true: 0.0



## 2. Ground-truth state-dependent diffusion

To make the stochastic problem nontrivial, the true noise level depends smoothly on the latent state.

We use a lower-triangular factor

\[
G^\star(z)
=
\begin{pmatrix}
g_{11}(z) & 0\\
g_{21}(z) & g_{22}(z)
\end{pmatrix}
\]

and define

\[
D^\star(z)=G^\star(z)G^\star(z)^\top.
\]

The diagonal entries remain positive everywhere.


In [3]:

def true_G(z):
    x = z[..., 0]
    y = z[..., 1]

    g11 = 0.18 + 0.08*torch.sigmoid(x)
    g21 = 0.05*torch.tanh(0.75*y)
    g22 = 0.13 + 0.07*torch.sigmoid(-y)

    zero = torch.zeros_like(g11)

    row1 = torch.stack([g11, zero], dim=-1)
    row2 = torch.stack([g21, g22], dim=-1)

    return torch.stack([row1, row2], dim=-2)

def true_D(z):
    Gm = true_G(z)
    return Gm @ Gm.transpose(-1, -2)

z_probe = torch.tensor(
    [[0.0, 0.0],
     [1.0, -1.0],
     [-1.0, 1.0]],
    device=DEVICE,
    dtype=DTYPE,
)

print("example G:")
print(true_G(z_probe))
print("example eig(D):")
print(torch.linalg.eigvalsh(true_D(z_probe)))


example G:
tensor([[[ 0.2200,  0.0000],
         [ 0.0000,  0.1650]],

        [[ 0.2385,  0.0000],
         [-0.0318,  0.1812]],

        [[ 0.2015,  0.0000],
         [ 0.0318,  0.1488]]])
example eig(D):
tensor([[0.0272, 0.0484],
        [0.0316, 0.0591],
        [0.0211, 0.0427]])



## 3. Generate stochastic trajectories

Euler–Maruyama:

\[
z_{t+\Delta t}
=
z_t+b^\star(z_t)\Delta t
+
G^\star(z_t)\sqrt{\Delta t}\,\varepsilon_t,
\qquad
\varepsilon_t\sim\mathcal N(0,I).
\]

Train and test trajectories are generated from independent initial conditions.

The model sees only pairs

\[
(z_t,z_{t+\Delta t}).
\]


In [4]:

DT = 0.04

N_TRAIN_PATHS = 320
N_TEST_PATHS = 80
STEPS_PER_PATH = 100

def sample_initial_states(n):
    return -2.0 + 4.0*torch.rand(
        n, 2,
        device=DEVICE,
        dtype=DTYPE,
    )

@torch.no_grad()
def generate_transition_dataset(n_paths, steps, seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    z = sample_initial_states(n_paths)

    z_now_all = []
    z_next_all = []
    path_id_all = []
    step_all = []

    sqrt_dt = math.sqrt(DT)

    for t in range(steps):
        b = true_drift(z)
        Gm = true_G(z)

        eps = torch.randn(
            n_paths, 2,
            device=DEVICE,
            dtype=DTYPE,
        )

        noise = torch.einsum("bij,bj->bi", Gm, eps)
        z_next = z + DT*b + sqrt_dt*noise

        z_now_all.append(z.clone())
        z_next_all.append(z_next.clone())
        path_id_all.append(torch.arange(n_paths, device=DEVICE))
        step_all.append(torch.full((n_paths,), t, device=DEVICE))

        z = z_next

    return {
        "z": torch.cat(z_now_all, dim=0),
        "z_next": torch.cat(z_next_all, dim=0),
        "path_id": torch.cat(path_id_all, dim=0),
        "step": torch.cat(step_all, dim=0),
    }

train_data = generate_transition_dataset(
    N_TRAIN_PATHS,
    STEPS_PER_PATH,
    seed=SEED+1,
)

test_data = generate_transition_dataset(
    N_TEST_PATHS,
    STEPS_PER_PATH,
    seed=SEED+2,
)

print("train transitions:", len(train_data["z"]))
print("test transitions:", len(test_data["z"]))
print("train state range:")
print(
    train_data["z"].min(dim=0).values.detach().cpu().numpy(),
    train_data["z"].max(dim=0).values.detach().cpu().numpy(),
)


train transitions: 32000
test transitions: 8000
train state range:
[-1.9630954 -1.9899526] [1.982198  1.9926093]



## 4. Model families

Both models use the same diffusion-network architecture.

### Plain SDE

\[
b_\theta(z)=\mathrm{MLP}_\theta(z).
\]

### Structured SDE

\[
b_\theta(z)
=
-M_\theta\nabla U_\theta(z)
+
A_\theta\nabla H_\theta(z),
\]

with

\[
M_\theta=LL^\top+\epsilon I
\]

and

\[
A_\theta^\top=-A_\theta.
\]

### Learned diffusion

The network returns a lower-triangular Cholesky factor

\[
G_\theta(z)
\]

with positive diagonal. Therefore

\[
D_\theta(z)=G_\theta G_\theta^\top
\]

is positive definite by construction.


In [5]:

class MLP(nn.Module):
    def __init__(self, din, dout, hidden=64, depth=2):
        super().__init__()
        layers = []
        d = din
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.Tanh()]
            d = hidden
        layers.append(nn.Linear(d, dout))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class ScalarPotential(nn.Module):
    def __init__(self, hidden=64, depth=2):
        super().__init__()
        self.net = MLP(2, 1, hidden=hidden, depth=depth)

    def forward(self, z):
        return self.net(z).squeeze(-1)

def grad_scalar(model, z, create_graph=True):
    z_req = z.requires_grad_(True)
    s = model(z_req).sum()
    return torch.autograd.grad(
        s,
        z_req,
        create_graph=create_graph,
        retain_graph=create_graph,
    )[0]

class PSDMatrix2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.raw = nn.Parameter(torch.tensor([0.2, 0.0, 0.2], dtype=DTYPE))

    def matrix(self):
        l11 = F.softplus(self.raw[0]) + 1e-3
        l21 = self.raw[1]
        l22 = F.softplus(self.raw[2]) + 1e-3

        L = torch.stack([
            torch.stack([l11, torch.zeros_like(l11)]),
            torch.stack([l21, l22]),
        ])

        eye = torch.eye(2, device=L.device, dtype=L.dtype)
        return L @ L.T + 1e-4*eye

class SkewMatrix2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.tensor(0.1, dtype=DTYPE))

    def matrix(self):
        zero = torch.zeros_like(self.a)
        return torch.stack([
            torch.stack([zero, -self.a]),
            torch.stack([self.a, zero]),
        ])

class DiffusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = MLP(2, 3, hidden=64, depth=2)

    def factor(self, z):
        raw = self.net(z)

        g11 = F.softplus(raw[..., 0]) + 1e-3
        g21 = raw[..., 1]
        g22 = F.softplus(raw[..., 2]) + 1e-3
        zero = torch.zeros_like(g11)

        row1 = torch.stack([g11, zero], dim=-1)
        row2 = torch.stack([g21, g22], dim=-1)

        return torch.stack([row1, row2], dim=-2)

    def covariance(self, z):
        Gm = self.factor(z)
        return Gm @ Gm.transpose(-1, -2)

class PlainSDE(nn.Module):
    def __init__(self):
        super().__init__()
        self.drift_net = MLP(2, 2, hidden=96, depth=3)
        self.diffusion = DiffusionNet()

    def drift(self, z, create_graph=True):
        return self.drift_net(z)

    def drift_components(self, z, create_graph=True):
        return None

    def diffusion_factor(self, z):
        return self.diffusion.factor(z)

class StructuredSDE(nn.Module):
    def __init__(self):
        super().__init__()
        self.U = ScalarPotential(hidden=64, depth=2)
        self.H = ScalarPotential(hidden=64, depth=2)
        self.M = PSDMatrix2D()
        self.A = SkewMatrix2D()
        self.diffusion = DiffusionNet()

    def components(self, z, create_graph=True):
        gu = grad_scalar(self.U, z, create_graph=create_graph)
        gh = grad_scalar(self.H, z, create_graph=create_graph)

        M = self.M.matrix()
        A = self.A.matrix()

        fd = -(gu @ M.T)
        fr = gh @ A.T

        return fd, fr, gu, gh, M, A

    def drift(self, z, create_graph=True):
        fd, fr, *_ = self.components(z, create_graph=create_graph)
        return fd + fr

    def drift_components(self, z, create_graph=True):
        return self.components(z, create_graph=create_graph)

    def diffusion_factor(self, z):
        return self.diffusion.factor(z)



## 5. Transition likelihood

For small \(\Delta t\),

\[
\Delta z
=
z_{t+\Delta t}-z_t
\]

is approximately Gaussian:

\[
\Delta z
\mid z_t
\sim
\mathcal N
\left(
b_\theta(z_t)\Delta t,\,
D_\theta(z_t)\Delta t
\right).
\]

We therefore train drift and diffusion jointly by minimizing the Gaussian negative log-likelihood.

Unlike a simple trajectory MSE, this objective cannot improve merely by predicting the conditional mean: it must also explain the conditional covariance.


In [6]:

LOG_2PI = math.log(2.0*math.pi)

def transition_nll(model, z, z_next, create_graph=True):
    b = model.drift(z, create_graph=create_graph)
    Gm = model.diffusion_factor(z)

    mean_inc = DT*b
    residual = (z_next-z) - mean_inc

    # Covariance of the increment.
    Sigma = DT*(Gm @ Gm.transpose(-1, -2))
    eye = torch.eye(2, device=z.device, dtype=z.dtype)
    Sigma = Sigma + 1e-6*eye

    chol = torch.linalg.cholesky(Sigma)

    sol = torch.cholesky_solve(
        residual.unsqueeze(-1),
        chol,
    ).squeeze(-1)

    maha = (residual*sol).sum(dim=-1)
    logdet = 2.0*torch.log(
        torch.diagonal(chol, dim1=-2, dim2=-1)
    ).sum(dim=-1)

    nll = 0.5*(maha + logdet + 2*LOG_2PI)
    return nll.mean()



## 6. Training

The structured model requires differentiating through \(\nabla U_\theta\) and \(\nabla H_\theta\), so it is computationally more expensive than the plain MLP.

On CPU, reduce `EPOCHS` to 500 for a quick smoke test.  
For the actual Phase 11B comparison use at least the default configuration.


In [ ]:

@dataclass
class TrainConfig:
    epochs: int = 1200
    batch_size: int = 512
    lr: float = 1.5e-3
    weight_decay: float = 1e-6
    print_every: int = 200

CFG = TrainConfig()

def train_sde(model, data, cfg=CFG):
    model = model.to(DEVICE)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    z_all = data["z"]
    zn_all = data["z_next"]
    n = len(z_all)

    hist = []

    for epoch in range(1, cfg.epochs+1):
        idx = torch.randint(
            0, n,
            (cfg.batch_size,),
            device=DEVICE,
        )

        z = z_all[idx].clone()
        zn = zn_all[idx]

        model.train()
        loss = transition_nll(
            model,
            z,
            zn,
            create_graph=True,
        )

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=10.0,
        )
        opt.step()

        hist.append(float(loss.detach().cpu()))

        if epoch == 1 or epoch % cfg.print_every == 0:
            print(
                f"{model.__class__.__name__}: "
                f"epoch={epoch:4d} "
                f"NLL={hist[-1]:.6f}"
            )

    return model, hist

plain_sde = PlainSDE()
structured_sde = StructuredSDE()

plain_sde, hist_plain = train_sde(
    plain_sde,
    train_data,
)

structured_sde, hist_struct = train_sde(
    structured_sde,
    train_data,
)


PlainSDE: epoch=   1 NLL=-1.985991
PlainSDE: epoch= 200 NLL=-3.718635
PlainSDE: epoch= 400 NLL=-3.695395
PlainSDE: epoch= 600 NLL=-3.738873
PlainSDE: epoch= 800 NLL=-3.689852
PlainSDE: epoch=1000 NLL=-3.754331
PlainSDE: epoch=1200 NLL=-3.718439
StructuredSDE: epoch=   1 NLL=-1.818096


In [ ]:

plt.figure(figsize=(7,4))
plt.plot(hist_plain, label="plain SDE")
plt.plot(hist_struct, label="structured SDE")
plt.xlabel("training step")
plt.ylabel("transition NLL")
plt.title("Phase 11B — stochastic transition likelihood")
plt.legend()
plt.show()



## 7. Independent field evaluation

We evaluate on fresh latent states—not only on states seen in the stochastic trajectories.

Metrics:

### Drift
\[
\mathrm{MSE}(b_\theta,b^\star).
\]

For the structured model we also measure component recovery:

\[
F_{\rm diss},\qquad F_{\rm rot}.
\]

### Diffusion
We compare covariance fields:

\[
D_\theta(z)
\quad\text{vs}\quad
D^\star(z).
\]

### Structural identities

\[
M_\theta\succeq0,
\qquad
A_\theta^\top=-A_\theta,
\]

\[
\nabla U_\theta^\top F_{\rm diss}\le0,
\]

\[
\nabla H_\theta^\top F_{\rm rot}=0.
\]


In [ ]:

def fresh_states(n=4000):
    return -2.0 + 4.0*torch.rand(
        n, 2,
        device=DEVICE,
        dtype=DTYPE,
    )

def mse_tensor(a, b):
    return float(F.mse_loss(a, b).detach().cpu())

def cosine_mean(a, b, eps=1e-8):
    num = (a*b).sum(dim=-1)
    den = (
        torch.norm(a, dim=-1)
        * torch.norm(b, dim=-1)
        + eps
    )
    return float((num/den).mean().detach().cpu())

def covariance_relative_error(Dhat, Dtrue, eps=1e-8):
    err = torch.linalg.matrix_norm(
        Dhat-Dtrue,
        ord="fro",
        dim=(-2,-1),
    )
    base = torch.linalg.matrix_norm(
        Dtrue,
        ord="fro",
        dim=(-2,-1),
    )
    return float((err/(base+eps)).mean().detach().cpu())

z_eval = fresh_states(4000)

with torch.no_grad():
    b_true_eval = true_drift(z_eval)
    fd_true_eval, fr_true_eval = true_drift_components(z_eval)
    D_true_eval = true_D(z_eval)

def eval_plain_sde(model):
    model.eval()

    with torch.no_grad():
        b = model.drift(z_eval)
        Gm = model.diffusion_factor(z_eval)
        D = Gm @ Gm.transpose(-1,-2)

    return {
        "model": "plain_sde",
        "drift_mse": mse_tensor(b, b_true_eval),
        "diffusion_D_mse": mse_tensor(D, D_true_eval),
        "diffusion_D_relative_fro": covariance_relative_error(D, D_true_eval),
    }

def eval_structured_sde(model):
    model.eval()

    z = z_eval.clone().requires_grad_(True)

    fd, fr, gu, gh, M, A = model.components(
        z,
        create_graph=False,
    )
    b = fd + fr

    Gm = model.diffusion_factor(z)
    D = Gm @ Gm.transpose(-1,-2)

    diss_rate = (gu*fd).sum(dim=-1)
    rot_rate = (gh*fr).sum(dim=-1)

    return {
        "model": "structured_sde",
        "drift_mse": mse_tensor(b, b_true_eval),
        "diss_component_mse": mse_tensor(fd, fd_true_eval),
        "rot_component_mse": mse_tensor(fr, fr_true_eval),
        "diss_component_cosine": cosine_mean(fd, fd_true_eval),
        "rot_component_cosine": cosine_mean(fr, fr_true_eval),
        "diffusion_D_mse": mse_tensor(D, D_true_eval),
        "diffusion_D_relative_fro": covariance_relative_error(D, D_true_eval),
        "M_min_eig": float(torch.linalg.eigvalsh(M).min().detach().cpu()),
        "A_skew_error": float(torch.norm(A+A.T).detach().cpu()),
        "dissipation_violation_fraction": float(
            (diss_rate > 1e-6).float().mean().detach().cpu()
        ),
        "mean_dU_dt_diss": float(diss_rate.mean().detach().cpu()),
        "mean_abs_dH_dt_rot": float(rot_rate.abs().mean().detach().cpu()),
        "learned_A_scalar": float(model.A.a.detach().cpu()),
    }

field_metrics = pd.DataFrame([
    eval_plain_sde(plain_sde),
    eval_structured_sde(structured_sde),
])

display(field_metrics)
field_metrics.to_csv(
    "phase11B_field_metrics.csv",
    index=False,
)



## 8. Held-out likelihood and whitened-residual calibration

If both drift and diffusion are correctly learned, then

\[
r_t
=
\frac{1}{\sqrt{\Delta t}}
G_\theta(z_t)^{-1}
\left[
\Delta z_t-b_\theta(z_t)\Delta t
\right]
\]

should behave approximately like

\[
r_t\sim\mathcal N(0,I).
\]

We therefore inspect:

- test NLL;
- residual mean;
- residual covariance;
- mean squared norm.

For a calibrated two-dimensional Gaussian:

\[
E[\|r\|^2]\approx2.
\]


In [ ]:

def whitened_residuals(model, data):
    model.eval()

    z = data["z"].clone()
    zn = data["z_next"]

    # Structured drift still requires local autograd w.r.t. z.
    if isinstance(model, StructuredSDE):
        z = z.requires_grad_(True)
        b = model.drift(z, create_graph=False)
    else:
        with torch.no_grad():
            b = model.drift(z)

    with torch.no_grad():
        Gm = model.diffusion_factor(z)

    residual_scaled = (
        (zn-z.detach()) - DT*b.detach()
    ) / math.sqrt(DT)

    # G is lower triangular by construction.
    white = torch.linalg.solve_triangular(
        Gm.detach(),
        residual_scaled.unsqueeze(-1),
        upper=False,
    ).squeeze(-1)

    return white

def calibration_metrics(model, data, name):
    # Evaluate NLL in chunks to avoid a large graph/memory spike.
    vals = []
    B = 2048

    for start in range(0, len(data["z"]), B):
        z = data["z"][start:start+B].clone()
        zn = data["z_next"][start:start+B]

        if isinstance(model, StructuredSDE):
            z = z.requires_grad_(True)
            val = transition_nll(
                model,
                z,
                zn,
                create_graph=False,
            )
        else:
            with torch.no_grad():
                val = transition_nll(
                    model,
                    z,
                    zn,
                    create_graph=False,
                )

        vals.append(float(val.detach().cpu()))

    white = whitened_residuals(model, data).detach().cpu().numpy()

    mean = white.mean(axis=0)
    cov = np.cov(white.T)
    mean_r2 = np.mean(np.sum(white**2, axis=1))

    return {
        "model": name,
        "test_nll": float(np.mean(vals)),
        "white_mean_0": mean[0],
        "white_mean_1": mean[1],
        "white_cov_00": cov[0,0],
        "white_cov_01": cov[0,1],
        "white_cov_11": cov[1,1],
        "white_cov_fro_from_I": float(
            np.linalg.norm(cov-np.eye(2), ord="fro")
        ),
        "white_mean_squared_norm": float(mean_r2),
    }

calibration_df = pd.DataFrame([
    calibration_metrics(
        plain_sde,
        test_data,
        "plain_sde",
    ),
    calibration_metrics(
        structured_sde,
        test_data,
        "structured_sde",
    ),
])

display(calibration_df)
calibration_df.to_csv(
    "phase11B_calibration_metrics.csv",
    index=False,
)



## 9. Visual field checks

The plots below are diagnostic only. Quantitative conclusions should use the metrics above.


In [ ]:

grid = torch.linspace(-2, 2, 21, device=DEVICE)
gx, gy = torch.meshgrid(grid, grid, indexing="xy")
zg = torch.stack(
    [gx.reshape(-1), gy.reshape(-1)],
    dim=-1,
)

with torch.no_grad():
    b_true_g = true_drift(zg)
    D_true_g = true_D(zg)

with torch.no_grad():
    b_plain_g = plain_sde.drift(zg)

zg_struct = zg.clone().requires_grad_(True)
b_struct_g = structured_sde.drift(
    zg_struct,
    create_graph=False,
).detach()

for title, vec in [
    ("true drift", b_true_g),
    ("plain learned drift", b_plain_g),
    ("structured learned drift", b_struct_g),
]:
    arr = vec.detach().cpu().numpy()
    pos = zg.detach().cpu().numpy()

    plt.figure(figsize=(6,6))
    plt.quiver(
        pos[:,0], pos[:,1],
        arr[:,0], arr[:,1],
    )
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.title(title)
    plt.axis("equal")
    plt.show()

# Trace(D) = local total diffusion strength.
with torch.no_grad():
    Gp = plain_sde.diffusion_factor(zg)
    Dp = Gp @ Gp.transpose(-1,-2)

Gs = structured_sde.diffusion_factor(zg_struct.detach())
Ds = Gs @ Gs.transpose(-1,-2)

for title, Dm in [
    ("true diffusion trace", D_true_g),
    ("plain diffusion trace", Dp),
    ("structured diffusion trace", Ds),
]:
    strength = torch.diagonal(
        Dm,
        dim1=-2,
        dim2=-1,
    ).sum(dim=-1).detach().cpu().numpy()

    plt.figure(figsize=(6,5))
    plt.scatter(
        zg[:,0].detach().cpu().numpy(),
        zg[:,1].detach().cpu().numpy(),
        c=strength,
        s=25,
    )
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.title(title)
    plt.colorbar(label="trace(D)")
    plt.show()



## 10. Ensemble rollout distribution

A stochastic model should reproduce distributions, not a single path.

Starting from the same initial state, we simulate an ensemble under:

- the true SDE;
- the plain learned SDE;
- the structured learned SDE.

We compare final ensemble mean and covariance.


In [ ]:

@torch.no_grad()
def true_sde_step(z, eps):
    b = true_drift(z)
    Gm = true_G(z)
    noise = torch.einsum("bij,bj->bi", Gm, eps)
    return z + DT*b + math.sqrt(DT)*noise

def learned_sde_step(model, z, eps):
    if isinstance(model, StructuredSDE):
        z_req = z.detach().requires_grad_(True)
        b = model.drift(z_req, create_graph=False).detach()
        with torch.no_grad():
            Gm = model.diffusion_factor(z_req.detach())
    else:
        with torch.no_grad():
            b = model.drift(z)
            Gm = model.diffusion_factor(z)

    noise = torch.einsum("bij,bj->bi", Gm, eps)
    return z.detach() + DT*b + math.sqrt(DT)*noise

def simulate_ensemble(kind, z0, eps_seq, model=None):
    n = eps_seq.shape[1]

    z = z0[None,:].repeat(n,1).to(DEVICE)

    for t in range(len(eps_seq)):
        eps = eps_seq[t]

        if kind == "true":
            z = true_sde_step(z, eps)
        else:
            z = learned_sde_step(model, z, eps)

    return z.detach()

def empirical_mean_cov(z):
    x = z.detach().cpu().numpy()
    mu = x.mean(axis=0)
    cov = np.cov(x.T)
    return mu, cov

N_ENSEMBLE = 384
N_ROLLOUT_STEPS = 80

torch.manual_seed(SEED+99)
eps_seq = torch.randn(
    N_ROLLOUT_STEPS,
    N_ENSEMBLE,
    2,
    device=DEVICE,
    dtype=DTYPE,
)

z0_list = [
    torch.tensor([1.6, 0.6], device=DEVICE, dtype=DTYPE),
    torch.tensor([-1.2, 1.4], device=DEVICE, dtype=DTYPE),
    torch.tensor([0.4, -1.5], device=DEVICE, dtype=DTYPE),
]

ensemble_rows = []

for ic, z0 in enumerate(z0_list):
    final_true = simulate_ensemble(
        "true",
        z0,
        eps_seq,
    )

    final_plain = simulate_ensemble(
        "learned",
        z0,
        eps_seq,
        model=plain_sde,
    )

    final_struct = simulate_ensemble(
        "learned",
        z0,
        eps_seq,
        model=structured_sde,
    )

    mu_true, cov_true = empirical_mean_cov(final_true)

    for name, final in [
        ("plain_sde", final_plain),
        ("structured_sde", final_struct),
    ]:
        mu, cov = empirical_mean_cov(final)

        ensemble_rows.append({
            "initial_condition": ic,
            "model": name,
            "final_mean_error": float(
                np.linalg.norm(mu-mu_true)
            ),
            "final_cov_fro_error": float(
                np.linalg.norm(cov-cov_true, ord="fro")
            ),
        })

    true_np = final_true.cpu().numpy()
    plain_np = final_plain.cpu().numpy()
    struct_np = final_struct.cpu().numpy()

    plt.figure(figsize=(6,6))
    plt.scatter(
        true_np[:,0],
        true_np[:,1],
        s=10,
        alpha=0.35,
        label="true",
    )
    plt.scatter(
        plain_np[:,0],
        plain_np[:,1],
        s=10,
        alpha=0.25,
        label="plain",
    )
    plt.scatter(
        struct_np[:,0],
        struct_np[:,1],
        s=10,
        alpha=0.25,
        label="structured",
    )
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.title(f"Final ensemble distribution — IC {ic}")
    plt.axis("equal")
    plt.legend()
    plt.show()

ensemble_df = pd.DataFrame(ensemble_rows)
display(ensemble_df)

ensemble_df.to_csv(
    "phase11B_ensemble_metrics.csv",
    index=False,
)



## 11. Drift / diffusion separation diagnostics

The key scientific question is not merely whether transition likelihood is good.

We want to know whether:

1. the conditional mean is represented by the learned drift;
2. the conditional covariance is represented by \(D_\theta\);
3. the structured drift retains its dissipative/rotational interpretation;
4. diffusion is not being inflated to hide systematic drift errors.

A useful warning sign is:

\[
\text{good NLL}
\quad\text{but}\quad
\text{poor drift MSE}.
\]

That would indicate partial **drift–diffusion confounding**.


In [ ]:

summary = (
    field_metrics
    .merge(
        calibration_df,
        on="model",
        how="left",
    )
)

display(summary)
summary.to_csv(
    "phase11B_summary.csv",
    index=False,
)

print("saved:")
print(" - phase11B_field_metrics.csv")
print(" - phase11B_calibration_metrics.csv")
print(" - phase11B_ensemble_metrics.csv")
print(" - phase11B_summary.csv")



# Phase 11B decision gate

The experiment supports moving forward only if the following pattern is observed.

### A. Drift is learned

Both learned models should have low drift error, and the structured model should remain reasonably competitive with the plain model.

### B. Structured components remain meaningful

For the structured model:

\[
\cos(F_{\rm diss}^\theta,F_{\rm diss}^\star)
\]

and

\[
\cos(F_{\rm rot}^\theta,F_{\rm rot}^\star)
\]

should remain high.

The identities

\[
\nabla U_\theta^\top F_{\rm diss}\le0
\]

and

\[
\nabla H_\theta^\top F_{\rm rot}\approx0
\]

should still hold.

### C. Diffusion is recovered

The covariance error

\[
\|D_\theta-D^\star\|
\]

should be small, and whitened residual covariance should be close to \(I\).

### D. Distributional rollout works

Long-run ensemble mean and covariance should track the true SDE.

### E. No evidence of severe drift–diffusion confounding

Good likelihood should not coexist with a badly wrong drift.

---

## If Phase 11B passes

The next phase should no longer be merely another synthetic SDE.

A sensible Phase 12 is to test the decomposition on a learned latent representation, for example

\[
x
\overset{\text{encoder}}{\longrightarrow}
z
\overset{\text{structured SDE}}{\longrightarrow}
z'
\overset{\text{decoder}}{\longrightarrow}
x',
\]

or on a small generative trajectory task where the structured and plain models are compared at matched parameter/compute budget.

Before making a strong claim, repeat Phase 11B over multiple random seeds.
